# 03 — Modèle Random Forest (Forêt Aléatoire)
## Projet EDF — Prédiction de la Consommation Électrique

**Algorithme :** `RandomForestRegressor` (scikit-learn)

La forêt aléatoire est un ensemble de N arbres de décision entraînés sur des sous-échantillons aléatoires du dataset (bootstrapping = bagging). La prédiction finale est la **moyenne** des prédictions de tous les arbres.

**Avantages pour ce projet :**
- Robuste aux outliers et aux données bruitées
- Interprétable via l'importance des features (feature importance)
- Pas besoin de normalisation des features (non-paramétrique)
- Bonne généralisation sans sur-apprentissage grâce à l'ensemble

**Plan :**
1. Chargement des données prétraitées
2. Recherche des hyperparamètres (GridSearchCV)
3. Entraînement du modèle final
4. Évaluation (R², RMSE, MAPE, Accuracy)
5. Feature importance
6. Visualisation des prédictions
7. Sauvegarde du modèle

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import time
from pathlib import Path

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.inspection import permutation_importance

from models.evaluate import evaluate_model, r2_score, rmse, mape

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.dpi': 110})

PROCESSED_PATH = Path('../data/processed')
MODELS_PATH    = Path('../data/models_saved')
MODELS_PATH.mkdir(parents=True, exist_ok=True)

print('✅ Imports OK')

## 1. Chargement des données prétraitées

In [ ]:
X_train = np.load(PROCESSED_PATH / 'X_train.npy')
X_test  = np.load(PROCESSED_PATH / 'X_test.npy')
y_train = np.load(PROCESSED_PATH / 'y_train.npy')
y_test  = np.load(PROCESSED_PATH / 'y_test.npy')
feature_cols = pd.read_csv(PROCESSED_PATH / 'feature_cols.csv', header=None)[0].tolist()

print(f'X_train : {X_train.shape} | y_train : {y_train.shape}')
print(f'X_test  : {X_test.shape}  | y_test  : {y_test.shape}')
print(f'Features ({len(feature_cols)}) : {feature_cols}')

## 2. Recherche des hyperparamètres (GridSearchCV)

On utilise une validation croisée temporelle (TimeSeriesSplit) pour respecter la causalité des données.

In [ ]:
from sklearn.model_selection import TimeSeriesSplit

param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [10, 15, None],
    'min_samples_split': [5, 10],
    'min_samples_leaf': [3, 5],
}

tscv = TimeSeriesSplit(n_splits=5)

print('Lancement GridSearchCV...')
print(f'Combinaisons testées : {2 * 3 * 2 * 2} × 5 folds = {2*3*2*2*5} fits')

t0 = time.time()
gs = GridSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    param_grid,
    cv=tscv,
    scoring='neg_mean_absolute_percentage_error',
    n_jobs=-1,
    verbose=1
)
gs.fit(X_train, y_train)

print(f'\nDurée GridSearch : {time.time() - t0:.1f}s')
print(f'Meilleurs paramètres : {gs.best_params_}')
print(f'Meilleur MAPE CV    : {-gs.best_score_*100:.2f}%')

## 3. Entraînement du modèle final

In [ ]:
# Entraînement avec les meilleurs hyperparamètres sur tout le train set
best_params = gs.best_params_

t0 = time.time()
rf = RandomForestRegressor(
    n_estimators=best_params.get('n_estimators', 200),
    max_depth=best_params.get('max_depth', 15),
    min_samples_split=best_params.get('min_samples_split', 10),
    min_samples_leaf=best_params.get('min_samples_leaf', 5),
    max_features='sqrt',
    bootstrap=True,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)
training_time = time.time() - t0

print(f'Entraînement terminé en {training_time:.1f}s')
print(f'Nombre d\'arbres : {rf.n_estimators}')
print(f'Profondeur max  : {rf.max_depth}')

## 4. Évaluation des performances

In [ ]:
metrics_rf = evaluate_model(rf, X_test, y_test, model_name='Random Forest')
print(f'\nTemps d\'entraînement : {training_time:.1f}s')

In [ ]:
# Vérification du sur-apprentissage (train vs test)
y_pred_train = rf.predict(X_train)
y_pred_test  = rf.predict(X_test)

print('Comparaison Train / Test (sur-apprentissage) :')
print(f'  R² Train : {r2_score(y_train, y_pred_train):.4f}')
print(f'  R² Test  : {r2_score(y_test, y_pred_test):.4f}')
print(f'  MAPE Train : {mape(y_train, y_pred_train):.2f}%')
print(f'  MAPE Test  : {mape(y_test, y_pred_test):.2f}%')

gap = r2_score(y_train, y_pred_train) - r2_score(y_test, y_pred_test)
if gap < 0.05:
    print('\n✅ Pas de sur-apprentissage significatif (gap R² < 0.05)')
else:
    print(f'\n⚠️  Sur-apprentissage détecté (gap R² = {gap:.3f})')

## 5. Importance des features

In [ ]:
# Feature importance intégrée (MDI — Mean Decrease Impurity)
importances = pd.Series(rf.feature_importances_, index=feature_cols)
importances = importances.sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 6))
colors = ['steelblue' if v < 0.20 else 'coral' for v in importances.values]
bars = ax.barh(importances.index, importances.values * 100, color=colors, alpha=0.85)
ax.set_xlabel('Importance relative (%)')
ax.set_title('Importance des features — Random Forest\n(MDI — Mean Decrease Impurity)', fontweight='bold')

# Annotations
for bar, val in zip(bars, importances.values):
    ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
            f'{val*100:.1f}%', va='center', fontsize=9)

ax.axvline(100/len(feature_cols), color='red', linestyle='--',
            alpha=0.6, label=f'Importance uniforme ({100/len(feature_cols):.1f}%)')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(PROCESSED_PATH / '07_rf_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top 5 features :')
print(importances.sort_values(ascending=False).head())

## 6. Visualisation des prédictions

In [ ]:
df_proc = pd.read_parquet(PROCESSED_PATH / 'dataset_processed.parquet')
split_idx = len(y_train)
dates_test = df_proc['date'].values[split_idx:split_idx + len(y_test)]

fig, axes = plt.subplots(3, 1, figsize=(15, 12))
fig.suptitle('Random Forest — Analyse des Prédictions vs Réalité', fontweight='bold', fontsize=13)

# ── Panel 1 : Série temporelle
ax1 = axes[0]
ax1.plot(dates_test, y_test / 1000, label='Réalité', linewidth=1.2,
          color='steelblue', alpha=0.8)
ax1.plot(dates_test, y_pred_test / 1000, label='Prédiction RF', linewidth=1.2,
          color='coral', linestyle='--', alpha=0.85)
ax1.set_ylabel('Consommation (GW)')
ax1.set_title('Série temporelle — Prédiction vs Réalité (jeu de test)')
ax1.legend()

# ── Panel 2 : Scatter Réalité vs Prédiction
ax2 = axes[1]
ax2.scatter(y_test / 1000, y_pred_test / 1000, alpha=0.4, s=10, color='steelblue')
lims = [min(y_test.min(), y_pred_test.min()) / 1000,
         max(y_test.max(), y_pred_test.max()) / 1000]
ax2.plot(lims, lims, 'r--', linewidth=1.5, label='Prédiction parfaite')
ax2.set_xlabel('Réalité (GW)'); ax2.set_ylabel('Prédiction (GW)')
ax2.set_title(f'Scatter — R² = {metrics_rf["r2"]:.4f} | MAPE = {metrics_rf["mape_pct"]:.2f}%')
ax2.legend()

# ── Panel 3 : Distribution des résidus
ax3 = axes[2]
residus = (y_test - y_pred_test) / 1000
ax3.hist(residus, bins=50, color='steelblue', alpha=0.8, edgecolor='white')
ax3.axvline(0, color='red', linestyle='--', linewidth=1.5, label='Erreur = 0')
ax3.axvline(residus.mean(), color='orange', linestyle='--',
             label=f'Biais moyen : {residus.mean():.2f} GW')
ax3.set_xlabel('Résidu (GW)')
ax3.set_title('Distribution des résidus (prédiction - réalité)')
ax3.legend()

plt.tight_layout()
plt.savefig(PROCESSED_PATH / '08_rf_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Sauvegarde du modèle

In [ ]:
model_path = MODELS_PATH / 'random_forest_v1.pkl'
joblib.dump(rf, model_path)

# Sauvegarde des métriques
metrics_rf['training_time_s'] = round(training_time, 1)
metrics_rf['hyperparameters'] = best_params
pd.Series(metrics_rf).to_json(MODELS_PATH / 'metrics_random_forest.json')

print(f'✅ Modèle sauvegardé → {model_path}')
print(f'   Taille du fichier : {model_path.stat().st_size / 1024 / 1024:.1f} Mo')

# Vérification du chargement
rf_loaded = joblib.load(model_path)
y_test_check = rf_loaded.predict(X_test[:3])
print(f'\n✅ Vérification chargement OK — Prédictions test (3 premières) :')
for i, pred in enumerate(y_test_check):
    print(f'   Prédiction {i+1}: {pred:,.0f} MW (Réalité: {y_test[i]:,.0f} MW)')

print(f'\n📊 RÉSUMÉ Random Forest :')
print(f'   R² = {metrics_rf["r2"]}  |  MAPE = {metrics_rf["mape_pct"]}%  |  RMSE = {metrics_rf["rmse_mw"]:,.0f} MW')